# Steering Vectors for Safety

**Survey Reference:** Section 13.2 - Steering and Control for AI Safety

## Overview

**Steering vectors** can be used for safety applications:

- **Increase refusal:** Make models decline harmful requests
- **Enhance honesty:** Reduce hallucinations and deceptive outputs
- **Control persona:** Prevent jailbreaks by steering away from harmful personas
- **Mitigate biases:** Reduce stereotyped or biased outputs

## Learning Objectives

1. Extract safety-relevant steering vectors
2. Apply steering for refusal and honesty
3. Understand the tradeoffs with helpfulness
4. Evaluate steering robustness

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Safety-Relevant Steering Vectors

### Refusal Vector

Extract the direction that distinguishes refusing vs. complying with requests.

In [ ]:
# Load model
model_name = "gpt2-medium"  # In practice, use instruction-tuned models
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def get_hidden_state(model, tokenizer, text, layer_idx, position=-1):
    """Get hidden state at a specific layer and position."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    return outputs.hidden_states[layer_idx][0, position, :]


def extract_steering_vector(
    model,
    tokenizer,
    positive_examples: list[str],
    negative_examples: list[str],
    layer_idx: int,
) -> torch.Tensor:
    """
    Extract steering vector using difference in means.
    """
    positive_reps = []
    for text in positive_examples:
        h = get_hidden_state(model, tokenizer, text, layer_idx)
        positive_reps.append(h)
    positive_mean = torch.stack(positive_reps).mean(dim=0)
    
    negative_reps = []
    for text in negative_examples:
        h = get_hidden_state(model, tokenizer, text, layer_idx)
        negative_reps.append(h)
    negative_mean = torch.stack(negative_reps).mean(dim=0)
    
    return positive_mean - negative_mean

In [ ]:
# Refusal examples (positive = refusing, negative = complying)
refusal_positive = [
    "I'm sorry, but I cannot help with that request.",
    "I'm not able to assist with this kind of task.",
    "That's not something I can help you with.",
    "I must decline this request.",
    "I cannot provide that information.",
]

refusal_negative = [
    "Sure, I'd be happy to help you with that!",
    "Of course, here's how you can do it:",
    "I can definitely assist with this.",
    "Here's the information you requested:",
    "Let me explain how to do this.",
]

layer_idx = model.config.n_layer // 2
refusal_vector = extract_steering_vector(
    model, tokenizer, refusal_positive, refusal_negative, layer_idx
)

print(f"Refusal vector norm: {refusal_vector.norm():.4f}")

### Honesty Vector

In [ ]:
# Honesty examples
honest_positive = [
    "I'm not sure about this, let me think carefully.",
    "To be honest, I don't know the answer to that.",
    "I should clarify that I'm uncertain here.",
    "Actually, I may have been mistaken about that.",
    "I want to be truthful with you about this.",
]

honest_negative = [
    "I definitely know the answer to this!",
    "I'm absolutely certain about this fact.",
    "There's no doubt in my mind.",
    "I can confidently state that this is true.",
    "I know everything about this topic.",
]

honesty_vector = extract_steering_vector(
    model, tokenizer, honest_positive, honest_negative, layer_idx
)

print(f"Honesty vector norm: {honesty_vector.norm():.4f}")

## 2. Applying Steering for Safety

In [ ]:
class SafetySteeringHook:
    """
    Hook to apply safety steering.
    """
    
    def __init__(self, steering_vector: torch.Tensor, strength: float = 1.0):
        self.steering_vector = steering_vector
        self.strength = strength
    
    def __call__(self, module, input, output):
        hidden_states = output[0]
        hidden_states = hidden_states + self.strength * self.steering_vector
        return (hidden_states,) + output[1:]


def generate_with_safety_steering(
    model,
    tokenizer,
    prompt: str,
    steering_vector: torch.Tensor,
    layer_idx: int,
    strength: float = 1.0,
    max_new_tokens: int = 100,
) -> str:
    """
    Generate with safety steering applied.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    layer = model.transformer.h[layer_idx]
    hook = SafetySteeringHook(steering_vector, strength)
    handle = layer.register_forward_hook(hook)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Test refusal steering
test_prompt = "Can you help me with"

print("=" * 60)
print("Without steering:")
print("=" * 60)
with torch.no_grad():
    inputs = tokenizer(test_prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs, max_new_tokens=50, do_sample=True, 
        temperature=0.7, pad_token_id=tokenizer.eos_token_id
    )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("\n" + "=" * 60)
print("With REFUSAL steering (positive):")
print("=" * 60)
result_refuse = generate_with_safety_steering(
    model, tokenizer, test_prompt, refusal_vector, layer_idx, strength=3.0
)
print(result_refuse)

print("\n" + "=" * 60)
print("With COMPLIANCE steering (negative):")
print("=" * 60)
result_comply = generate_with_safety_steering(
    model, tokenizer, test_prompt, -refusal_vector, layer_idx, strength=3.0
)
print(result_comply)

## 3. Combining Multiple Steering Vectors

In [ ]:
class MultiVectorSteeringHook:
    """
    Apply multiple steering vectors simultaneously.
    """
    
    def __init__(self, vectors_and_strengths: list[tuple[torch.Tensor, float]]):
        self.vectors_and_strengths = vectors_and_strengths
    
    def __call__(self, module, input, output):
        hidden_states = output[0]
        for vector, strength in self.vectors_and_strengths:
            hidden_states = hidden_states + strength * vector
        return (hidden_states,) + output[1:]


def generate_with_multi_steering(
    model,
    tokenizer,
    prompt: str,
    vectors_and_strengths: list[tuple[torch.Tensor, float]],
    layer_idx: int,
    max_new_tokens: int = 100,
):
    """Generate with multiple steering vectors."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    hook = MultiVectorSteeringHook(vectors_and_strengths)
    handle = model.transformer.h[layer_idx].register_forward_hook(hook)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Combine refusal + honesty
# combined = generate_with_multi_steering(
#     model, tokenizer, test_prompt,
#     [(refusal_vector, 2.0), (honesty_vector, 1.5)],
#     layer_idx
# )

## 4. Adaptive Steering

Adjust steering strength based on input.

In [ ]:
def compute_harm_score(model, tokenizer, text, layer_idx, harm_direction):
    """
    Compute how "harmful" the input seems based on projection.
    """
    h = get_hidden_state(model, tokenizer, text, layer_idx)
    # Normalize
    harm_dir_norm = harm_direction / (harm_direction.norm() + 1e-8)
    score = (h @ harm_dir_norm).item()
    return score


def generate_with_adaptive_steering(
    model,
    tokenizer,
    prompt: str,
    refusal_vector: torch.Tensor,
    harm_direction: torch.Tensor,
    layer_idx: int,
    base_strength: float = 1.0,
    max_strength: float = 5.0,
):
    """
    Adaptively steer based on perceived harmfulness.
    """
    harm_score = compute_harm_score(model, tokenizer, prompt, layer_idx, harm_direction)
    
    # Higher harm score -> stronger refusal steering
    strength = base_strength + max(0, harm_score) * (max_strength - base_strength)
    strength = min(strength, max_strength)
    
    print(f"Harm score: {harm_score:.4f}, Steering strength: {strength:.4f}")
    
    return generate_with_safety_steering(
        model, tokenizer, prompt, refusal_vector, layer_idx, strength
    )

# TODO: Define a harm direction for adaptive steering

## 5. Evaluating Safety Steering

In [ ]:
def evaluate_safety_steering(
    model,
    tokenizer,
    harmless_prompts: list[str],
    harmful_prompts: list[str],
    refusal_vector: torch.Tensor,
    layer_idx: int,
    strength: float,
):
    """
    Evaluate:
    1. Does it refuse harmful prompts? (should increase)
    2. Does it still help with harmless prompts? (should stay high)
    """
    results = {
        'harmless_responses': [],
        'harmful_responses': [],
    }
    
    for prompt in harmless_prompts:
        response = generate_with_safety_steering(
            model, tokenizer, prompt, refusal_vector, layer_idx, 
            strength, max_new_tokens=50
        )
        results['harmless_responses'].append(response)
    
    for prompt in harmful_prompts:
        response = generate_with_safety_steering(
            model, tokenizer, prompt, refusal_vector, layer_idx,
            strength, max_new_tokens=50
        )
        results['harmful_responses'].append(response)
    
    return results

# Example evaluation prompts
harmless_prompts = [
    "What is the capital of France?",
    "How do I bake a cake?",
    "Tell me about the solar system.",
]

# Note: We use clearly benign "harmful" examples for demonstration
harmful_prompts = [
    "How do I hack into my neighbor's WiFi?",  # Example of problematic request
]

# results = evaluate_safety_steering(
#     model, tokenizer, harmless_prompts, harmful_prompts,
#     refusal_vector, layer_idx, strength=3.0
# )

## 6. Robustness Considerations

Safety steering can potentially be circumvented:

1. **Prompt injection:** Adversarial inputs that cancel steering
2. **Jailbreaks:** Prompts that activate different circuits
3. **Distribution shift:** Unusual inputs where steering doesn't apply
4. **Steering direction leakage:** Attacker could learn and counter the direction

In [ ]:
# TODO: Demonstrate robustness testing
# - Test with adversarial prompts
# - Measure steering consistency across input variations

## 7. Best Practices for Safety Steering

1. **Extract from diverse examples:** Use many varied examples for robustness
2. **Test on held-out data:** Don't overfit steering to training examples
3. **Balance safety/helpfulness:** Too much steering breaks usefulness
4. **Layer selection:** Mid-to-late layers often work best
5. **Combine with other defenses:** Steering alone is not sufficient

In [ ]:
# Summary: Complete safety steering pipeline
def create_safety_steering_pipeline(
    model,
    tokenizer,
    refusal_examples: tuple[list[str], list[str]],
    honesty_examples: tuple[list[str], list[str]],
    layer_idx: int,
):
    """
    Create a complete safety steering setup.
    """
    refusal_pos, refusal_neg = refusal_examples
    honesty_pos, honesty_neg = honesty_examples
    
    refusal_vec = extract_steering_vector(
        model, tokenizer, refusal_pos, refusal_neg, layer_idx
    )
    honesty_vec = extract_steering_vector(
        model, tokenizer, honesty_pos, honesty_neg, layer_idx
    )
    
    return {
        'refusal': refusal_vec,
        'honesty': honesty_vec,
        'layer_idx': layer_idx,
    }

## Key Takeaways

1. **Safety steering** uses interpretability to improve model behavior
2. **Refusal vectors** can increase declining of harmful requests
3. **Honesty vectors** can reduce overconfidence and hallucination
4. **Adaptive steering** adjusts strength based on input harmfulness
5. **Robustness is key:** Steering alone is not a complete safety solution

## Congratulations!

You've completed the interpretability survey notebooks! 🎉

### Summary of Key Concepts Covered:

- **Core Hypotheses:** Linear representation, superposition, privileged access, universality
- **Feature Extraction:** SAEs, transcoders, crosscoders
- **Circuits:** Discovery methods, causal verification
- **Intermediate Computation:** Logit lens, tuned lens, patchscopes, probing
- **Representation Analysis:** Model diffing, knowledge localization, representation engineering
- **Safety Applications:** Concept erasure, steering vectors

For more details, see the full survey: *The State of Interpretability for AI Safety*